# 🏛️ STF Judicial Lakehouse — Exploratory Data Analysis

This notebook demonstrates reproducible analytical exploration of the **STF Transparency Platform** dimensional lakehouse using **DuckDB** and **Polars**.

### Core Analytical Questions Explored:
1. **Annual Case Intake & Evolution**: How has case volume changed over time?
2. **Decision Dynamics**: What is the proportion of Monocratic vs. Collegiate rulings?
3. **Procedural Classes**: Which actions (ADI, RE, HC, ARE) dominate the court docket?
4. **Judicial Lead Time**: How many days elapsed between distribution and decision?
5. **Geographic Distribution**: Which Brazilian states and regions originate the most cases?

In [ ]:
import duckdb
import polars as pl
from pathlib import Path

# Connect zero-copy to the DuckDB Curated Lakehouse
db_path = Path("../../data/curated/stf_warehouse.duckdb").resolve()
con = duckdb.connect(str(db_path), read_only=True)
print(f"Connected to DuckDB: {db_path}")

## 1. Lakehouse Catalog & Tables
Let's list all registered Fact and Dimension tables.

In [ ]:
tables = con.execute("SHOW TABLES;").fetchall()
print("Available Lakehouse Models:")
for t in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
    print(f"- {t[0]:<20}: {count:,} rows")

## 2. Annual Process Inflow & Backlog Trends

In [ ]:
df_yearly = con.execute("""
    SELECT 
        d.year AS ano,
        COUNT(*) AS total_distribuidos,
        COUNT(CASE WHEN p.situacao = 'EM TRAMITAÇÃO' THEN 1 END) AS em_tramitacao,
        COUNT(CASE WHEN p.situacao = 'BAIXADO' THEN 1 END) AS baixados,
        COUNT(CASE WHEN p.situacao = 'JULGADO' THEN 1 END) AS julgados
    FROM fact_processes p
    JOIN dim_date d ON p.date_distribuicao_key = d.date_key
    GROUP BY d.year
    ORDER BY ano ASC;
""").pl()

print(df_yearly)

## 3. Decision Profile: Monocratic vs. Collegiate Rulings

In [ ]:
df_dec_profile = con.execute("""
    SELECT 
        categoria_decisao,
        COUNT(*) AS total_decisoes,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM fact_decisions
    GROUP BY categoria_decisao
    ORDER BY total_decisoes DESC;
""").pl()

print(df_dec_profile)

## 4. Top Judicial Classes

In [ ]:
df_classes = con.execute("""
    SELECT 
        classe_sigla,
        classe_descricao,
        COUNT(*) AS total_processos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM fact_processes
    GROUP BY classe_sigla, classe_descricao
    ORDER BY total_processos DESC
    LIMIT 10;
""").pl()

print(df_classes)

## 5. Judicial Lead Time (Days from Distribution to Decision)

In [ ]:
df_lead_time = con.execute("""
    SELECT 
        p.classe_sigla,
        ROUND(AVG(f.data_decisao - p.data_distribuicao), 1) AS media_dias,
        MEDIAN(f.data_decisao - p.data_distribuicao) AS mediana_dias,
        MIN(f.data_decisao - p.data_distribuicao) AS min_dias,
        MAX(f.data_decisao - p.data_distribuicao) AS max_dias
    FROM fact_decisions f
    JOIN fact_processes p ON f.process_id = p.process_id
    GROUP BY p.classe_sigla
    ORDER BY media_dias DESC;
""").pl()

print(df_lead_time)

## 6. Geographic Distribution Across Brazilian Regions

In [ ]:
df_geo = con.execute("""
    SELECT 
        o.regiao,
        COUNT(*) AS total_processos,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentual
    FROM fact_processes p
    JOIN dim_origin o ON p.uf_origem = o.uf_origem
    GROUP BY o.regiao
    ORDER BY total_processos DESC;
""").pl()

print(df_geo)

### Conclusion
The dimensional model in DuckDB provides sub-millisecond aggregations over 100% normalized STF public data. All queries maintain traceable data lineage back to official Corte Aberta files.